# 05. Ручное обогащение набора изображений

| Параметр | Значение |
|---|---|
| Данные | 54 фотографии автомобилей, Wikimedia Commons; дата обращения 01.11.2026 |
| Скрипты | `src/build_car_enrichment.py`, `src/contact_sheet.py`, `src/finalize_enrichment.py` |
| Исполнители | Якунченков К. А., Графов Г. П. |

Содержание: состав по стратам условий съёмки, причины отбраковки, лицензии и авторы.

## 0. Окружение

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pd.set_option("display.width", 170)
img = pd.read_csv(ROOT / "data/interim/car_enrichment/car_enrichment_manifest.csv")
acc = img[img.keep]
print(f"Кандидатов: {len(img)}; принято: {len(acc)} ({img.keep.mean():.0%}); в пилоте: {int(img.pilot.sum())}")

Кандидатов: 98; принято: 54 (55%); в пилоте: 30


## 1. Состав и отбраковка

Кандидаты получены поиском Wikimedia Commons по пяти стратам условий съёмки. Решение о включении принято
по контактным листам; причина отказа записана в манифест.

In [2]:
display(img.groupby(["stratum", "keep"]).size().unstack(fill_value=0)
        .rename(columns={False: "отказ", True: "принято"}))
display(img[~img.keep].reject_reason.value_counts().rename("кадров").to_frame())

keep,отказ,принято
stratum,,
dense_traffic,3,18
night,10,10
rain_wet,9,11
snow_winter,6,10
unusual_view,16,5


,кадров
reject_reason,
автомобили отсутствуют,29
"автомобили неразличимы (световые следы, < 10 пикселей, размытие)",11
историческая уличная сцена: автомобили единичны и неразличимы в толпе,2
"та же сцена и тот же автомобиль, что в кандидате 26 (один group_id)",1
спутниковый снимок облачности,1


## 2. Лицензии и авторы

In [3]:
lic = acc.license.value_counts().rename("изображений").to_frame()
lic["доля, %"] = (lic["изображений"] / len(acc) * 100).round(1)
display(lic)
print(f"Доля CC BY-SA: {acc.share_alike.mean():.0%}")
print(f"Уникальных авторов: {acc.author.nunique()}; максимум кадров одного автора: {acc.author.value_counts().max()}")

,изображений,"доля, %"
license,,
CC BY-SA 2.0,20,37.0
CC BY-SA 4.0,20,37.0
CC BY-SA 3.0,5,9.3
CC BY 2.0,3,5.6
Public domain,3,5.6
CC BY 4.0,2,3.7
CC0,1,1.9


Доля CC BY-SA: 83%
Уникальных авторов: 40; максимум кадров одного автора: 7


## 3. Выводы

1. Принято 54 фотографии из 98 кандидатов; для каждой в манифесте указаны автор, лицензия, ссылка на файл,
   SHA-256 и перцептивный хеш.
2. Набор, производный от изображений под CC BY-SA, распространяется под CC BY-SA.
3. Ограничения: один источник изображений; страта «необычный ракурс» заполнена не полностью (раздел 1).
4. Приведённый состав относится ко всем 54 отобранным изображениям. Разметка первого пилота (30 изображений) и статус повторного пилота рассмотрены в `07_annotation_agreement.ipynb`.